In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###### CELL 1 — IMPORT LIBRARIES AND DEFINE PROTOTYPE PATHS


We are starting a new notebook, so we need to recreate the basic Python
environment and define the paths for the saved features from Notebook 1.

Notebook 1 produced:

    train_features.pt

This file contains:
    • 4,491 trained FiLM-conditioned feature vectors [4491, 768]
    • Ground-truth multi-label disease labels [4491, 8]
    • Quality scores
    • Filenames

In this notebook, we will transform these learned feature representations
into 8 disease prototypes.

The final prototype file will be:

    disease_prototypes.pt

Each prototype will have 768 dimensions:

    [8, 768]

corresponding to:

    N, D, G, C, A, H, M, O




---



In [ ]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES AND DEFINE PROTOTYPE PATHS
# ============================================================

import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

ROOT = "/content/drive/My Drive/Eye Disease/Dataset"

TRAIN_FEATURES_PATH = os.path.join(
    ROOT,
    "train_features.pt"
)

PROTOTYPE_SAVE_PATH = os.path.join(
    ROOT,
    "disease_prototypes.pt"
)

# ------------------------------------------------------------
# DISEASE CONFIGURATION
# ------------------------------------------------------------

LABEL_COLUMNS = [
    "N", "D", "G", "C",
    "A", "H", "M", "O"
]

NUM_CLASSES = len(LABEL_COLUMNS)
FEATURE_DIM = 768

print("Device:", device)
print("Number of diseases:", NUM_CLASSES)
print("Expected feature dimension:", FEATURE_DIM)

Device: cuda
Number of diseases: 8
Expected feature dimension: 768



###### CELL 2 — LOAD THE TRAINED FEATURE REPRESENTATIONS


We now load the output of Notebook 1.

IMPORTANT:

We are NOT loading raw images and we are NOT running ConvNeXt again.

Notebook 1 has already processed every Stage-1 training eye through:

    Fundus Image
        ↓
    Shared ConvNeXt-Tiny
        ↓
    Quality Embedding + FiLM
        ↓
    Final quality-conditioned feature [768]

Therefore, this file contains the learned representation Fᵢ required
for prototype construction.

We will inspect the stored tensors and confirm that the expected data
was successfully recovered before calculating any prototypes.





---



In [ ]:
# ============================================================
# CELL 2 — LOAD THE TRAINED FEATURE REPRESENTATIONS
# ============================================================

assert os.path.exists(TRAIN_FEATURES_PATH), (
    f"Feature file not found:\n{TRAIN_FEATURES_PATH}"
)

feature_data = torch.load(
    TRAIN_FEATURES_PATH,
    map_location="cpu",
    weights_only=False
)

# Extract stored components
train_features = feature_data["features"].float()
train_labels = feature_data["labels"].float()
train_quality_scores = feature_data["quality_scores"].float()
train_filenames = feature_data["filenames"]

print("=" * 70)
print("TRAINED FEATURE DATA LOADED")
print("=" * 70)

print("\nFeatures shape:", train_features.shape)
print("Labels shape:", train_labels.shape)
print("Quality-score shape:", train_quality_scores.shape)
print("Number of filenames:", len(train_filenames))
print("\nStored keys:", list(feature_data.keys()))
print("=" * 70)

TRAINED FEATURE DATA LOADED

Features shape: torch.Size([4491, 768])
Labels shape: torch.Size([4491, 8])
Quality-score shape: torch.Size([4491])
Number of filenames: 4491

Stored keys: ['filenames', 'features', 'labels', 'quality_scores', 'label_columns', 'feature_dim']



###### CELL 3 — VERIFY FEATURE DATA ALIGNMENT



Before building prototypes, every feature must still correspond to the
correct ground-truth label.

For prototype construction, row i must mean:

    train_features[i]
            ↕
    train_labels[i]
            ↕
    train_filenames[i]

We verify:

    1. All components contain exactly 4,491 samples.
    2. Features have 768 dimensions.
    3. Labels have 8 disease columns.
    4. All feature and quality values are finite.
    5. Labels contain valid binary values.

This protects us from constructing prototypes from misaligned or corrupted
feature-label pairs.




---



In [ ]:
# ============================================================
# CELL 3 — VERIFY FEATURE DATA ALIGNMENT
# ============================================================

num_samples = train_features.shape[0]

# Shape alignment
assert train_labels.shape[0] == num_samples
assert train_quality_scores.shape[0] == num_samples
assert len(train_filenames) == num_samples

# Expected dimensions
assert train_features.shape[1] == FEATURE_DIM
assert train_labels.shape[1] == NUM_CLASSES

# Numerical validity
assert torch.isfinite(train_features).all()
assert torch.isfinite(train_quality_scores).all()
assert torch.isfinite(train_labels).all()

# Labels must be binary
unique_label_values = torch.unique(train_labels)
assert set(unique_label_values.tolist()).issubset({0.0, 1.0})

print("=" * 70)
print("FEATURE-LABEL ALIGNMENT VERIFIED")
print("=" * 70)

print("\nTotal aligned samples:", num_samples)
print("Feature dimension:", train_features.shape[1])
print("Disease labels:", train_labels.shape[1])
print("Feature values finite:", True)
print("Quality values finite:", True)
print("Labels valid binary values:", unique_label_values.tolist())

print("\nSUCCESS: Prototype input data is fully aligned.")
print("=" * 70)

FEATURE-LABEL ALIGNMENT VERIFIED

Total aligned samples: 4491
Feature dimension: 768
Disease labels: 8
Feature values finite: True
Quality values finite: True
Labels valid binary values: [0.0, 1.0]

SUCCESS: Prototype input data is fully aligned.



###### CELL 4 — CALCULATE DISEASE-WISE PROTOTYPE MEMBERSHIP


ODIR is a multi-label dataset.

Therefore, we must NOT assign every eye to only one disease.

For each disease k, we identify all training eyes satisfying:

    yᵢₖ = 1

and count how many features will contribute to prototype Pₖ.

Mathematically:

    Nₖ = Σᵢ yᵢₖ

An eye with multiple positive labels contributes to multiple disease
prototype groups.

For example:

    D = 1 and O = 1

means its learned feature contributes to BOTH:

    P_D
    P_O

These counts are essential because each prototype will be the mean of
the features belonging to that disease.




---



In [ ]:
# ============================================================
# CELL 4 — CALCULATE DISEASE-WISE PROTOTYPE MEMBERSHIP
# ============================================================

# Number of positive training samples for each disease
prototype_counts = train_labels.sum(dim=0).long()

# Total labels carried by each sample
labels_per_sample = train_labels.sum(dim=1)

# Identify multi-label samples
multi_label_mask = labels_per_sample > 1

# Create a useful summary
prototype_membership_df = pd.DataFrame({
    "Disease": LABEL_COLUMNS,
    "Positive_Samples": prototype_counts.cpu().numpy()
})

print("=" * 70)
print("DISEASE PROTOTYPE MEMBERSHIP")
print("=" * 70)

display(prototype_membership_df)

print("\nTotal training eyes:", num_samples)
print("Multi-label eyes:", multi_label_mask.sum().item())
print(
    "Total positive disease assignments:",
    prototype_counts.sum().item()
)

assert torch.all(prototype_counts > 0), (
    "At least one disease has zero positive samples."
)

print("\nSUCCESS: Every disease has samples available for prototype creation.")
print("=" * 70)

DISEASE PROTOTYPE MEMBERSHIP


,Disease,Positive_Samples
0,N,1502
1,D,1491
2,G,269
3,C,286
4,A,215
5,H,140
6,M,226
7,O,1089



Total training eyes: 4491
Multi-label eyes: 697
Total positive disease assignments: 5218

SUCCESS: Every disease has samples available for prototype creation.


###### CELL 5 — BUILD THE 8 DISEASE PROTOTYPES


We now perform the actual Disease Prototype Memory construction.

For each disease k:

    Pₖ = (1 / Nₖ) × Σ Fᵢ

where the summation includes only samples for which:

    yᵢₖ = 1

Instead of using an inefficient Python loop over 4,491 individual
features, we use matrix multiplication.

Given:

    Labels      [4491, 8]
    Features    [4491, 768]

we calculate:

    Labelsᵀ × Features

which produces:

    [8, 768]

Each row contains the summed feature representation for one disease.

We then divide each row by that disease's positive sample count.

The result is the actual Disease Prototype Memory:

    disease_prototypes [8, 768]

---

In [ ]:
# ============================================================
# CELL 5 — BUILD THE 8 DISEASE PROTOTYPES
# ============================================================

# ------------------------------------------------------------
# SUM FEATURES FOR EACH DISEASE
#
# [8, 4491] @ [4491, 768] = [8, 768]
# ------------------------------------------------------------

prototype_feature_sums = (
    train_labels.T @ train_features
)

# ------------------------------------------------------------
# DIVIDE BY DISEASE-SPECIFIC SAMPLE COUNTS
# ------------------------------------------------------------

disease_prototypes = (
    prototype_feature_sums
    / prototype_counts.unsqueeze(1).float()
)

# Final expected shape
assert disease_prototypes.shape == (
    NUM_CLASSES,
    FEATURE_DIM
)

# Numerical validity
assert torch.isfinite(disease_prototypes).all()

print("=" * 70)
print("DISEASE PROTOTYPES CREATED")
print("=" * 70)

print("\nPrototype tensor shape:")
print(disease_prototypes.shape)

print("\nExpected:")
print(f"torch.Size([{NUM_CLASSES}, {FEATURE_DIM}])")

print("\nSUCCESS: 8 disease prototypes successfully constructed.")
print("=" * 70)

DISEASE PROTOTYPES CREATED

Prototype tensor shape:
torch.Size([8, 768])

Expected:
torch.Size([8, 768])

SUCCESS: 8 disease prototypes successfully constructed.


###### CELL 6 — INDEPENDENTLY VERIFY THE PROTOTYPE FORMULA


The previous cell used efficient matrix multiplication to calculate all
8 prototypes simultaneously.

Now we independently verify the result.

For every disease, we manually select:

    features where label == 1

and calculate:

    mean(selected_features)

This manual result must match the corresponding prototype created in
Cell 5.

This is an important correctness test because it verifies that the
matrix-based implementation exactly follows:

    Pₖ = mean(Fᵢ | yᵢₖ = 1)

We also confirm that multi-label samples are naturally included in every
prototype for which their ground-truth label is positive.

---

In [ ]:
# ============================================================
# CELL 6 — INDEPENDENTLY VERIFY THE PROTOTYPE FORMULA
# ============================================================

max_verification_error = 0.0
verification_results = []

for disease_index, disease_name in enumerate(LABEL_COLUMNS):

    # Select all samples positive for this disease
    disease_mask = (
        train_labels[:, disease_index] == 1
    )

    disease_features = train_features[disease_mask]

    # Manual prototype calculation
    manual_prototype = disease_features.mean(dim=0)

    # Compare with matrix-calculated prototype
    difference = torch.max(
        torch.abs(
            manual_prototype
            - disease_prototypes[disease_index]
        )
    ).item()

    max_verification_error = max(
        max_verification_error,
        difference
    )

    verification_results.append({
        "Disease": disease_name,
        "Samples": disease_features.shape[0],
        "Max_Absolute_Error": difference
    })

verification_df = pd.DataFrame(
    verification_results
)

print("=" * 70)
print("PROTOTYPE FORMULA VERIFICATION")
print("=" * 70)

display(verification_df)

print("\nMaximum verification error:")
print(max_verification_error)

assert max_verification_error < 1e-5

print(
    "\nSUCCESS: All prototypes exactly match independent "
    "disease-wise feature averaging."
)
print("=" * 70)

PROTOTYPE FORMULA VERIFICATION


,Disease,Samples,Max_Absolute_Error
0,N,1502,1.788139e-07
1,D,1491,1.192093e-07
2,G,269,1.192093e-07
3,C,286,2.384186e-07
4,A,215,1.788139e-07
5,H,140,1.192093e-07
6,M,226,1.788139e-07
7,O,1089,1.788139e-07



Maximum verification error:
2.384185791015625e-07

SUCCESS: All prototypes exactly match independent disease-wise feature averaging.


###### CELL 7 — SAVE THE DISEASE PROTOTYPE MEMORY


The 8 disease prototypes have now been constructed and computationally
verified.

We save not only the prototype vectors but also the information required
to interpret and reuse them safely in Notebook 3.

The saved file includes:

    • disease_prototypes  → [8, 768]
    • disease names       → N, D, G, C, A, H, M, O
    • prototype counts    → number of positive samples per disease
    • feature dimension   → 768
    • construction method → mean of ground-truth disease-positive features

This creates a reusable Disease Prototype Memory that can later provide
Keys and Values to the Disease-Aware Attention module.

---

In [ ]:
# ============================================================
# CELL 7 — SAVE THE DISEASE PROTOTYPE MEMORY
# ============================================================

prototype_memory = {
    "disease_prototypes": disease_prototypes.cpu(),
    "label_columns": LABEL_COLUMNS,
    "prototype_counts": prototype_counts.cpu(),
    "feature_dim": FEATURE_DIM,
    "num_prototypes": NUM_CLASSES,
    "construction_method": (
        "Mean of FiLM-conditioned trained features "
        "for ground-truth disease-positive samples"
    )
}

torch.save(
    prototype_memory,
    PROTOTYPE_SAVE_PATH
)

# Verify saved file immediately
assert os.path.exists(PROTOTYPE_SAVE_PATH)

saved_prototype_memory = torch.load(
    PROTOTYPE_SAVE_PATH,
    map_location="cpu",
    weights_only=False
)

assert saved_prototype_memory[
    "disease_prototypes"
].shape == (NUM_CLASSES, FEATURE_DIM)

print("=" * 70)
print("DISEASE PROTOTYPE MEMORY SAVED")
print("=" * 70)

print("\nPath:")
print(PROTOTYPE_SAVE_PATH)

print("\nPrototype shape:")
print(
    saved_prototype_memory[
        "disease_prototypes"
    ].shape
)

print("\nDiseases:")
print(saved_prototype_memory["label_columns"])

print("\nSUCCESS: Disease Prototype Memory is saved and reusable.")
print("=" * 70)

DISEASE PROTOTYPE MEMORY SAVED

Path:
/content/drive/My Drive/Eye Disease/Dataset/disease_prototypes.pt

Prototype shape:
torch.Size([8, 768])

Diseases:
['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

SUCCESS: Disease Prototype Memory is saved and reusable.



###### CELL 8 — RELOAD AND VERIFY SAVED DISEASE PROTOTYPE MEMORY


The Disease Prototype Memory has been saved successfully.

Before analysing it further, we reload the saved file from Drive rather
than continuing only with the in-memory variable.

This verifies that:

    • the file was saved correctly
    • all expected metadata was preserved
    • the saved prototypes still have shape [8, 768]
    • the disease ordering is correct

The disease ordering is especially important because later attention
modules must know exactly which prototype corresponds to:

    N, D, G, C, A, H, M, O

We will use the reloaded prototype tensor for all subsequent analysis.


---

In [ ]:
# ============================================================
# CELL 8 — RELOAD AND VERIFY SAVED DISEASE PROTOTYPE MEMORY
# ============================================================

saved_memory = torch.load(
    PROTOTYPE_SAVE_PATH,
    map_location="cpu",
    weights_only=False
)

saved_prototypes = saved_memory[
    "disease_prototypes"
].float()

saved_labels = saved_memory["label_columns"]
saved_counts = saved_memory["prototype_counts"].long()

# Structural verification
assert saved_prototypes.shape == (
    NUM_CLASSES,
    FEATURE_DIM
)

assert saved_labels == LABEL_COLUMNS
assert torch.equal(saved_counts, prototype_counts)
assert torch.isfinite(saved_prototypes).all()

# Confirm saved tensor matches the one constructed before saving
save_reload_error = torch.max(
    torch.abs(
        saved_prototypes - disease_prototypes.cpu()
    )
).item()

assert save_reload_error == 0.0

print("=" * 70)
print("SAVED PROTOTYPE MEMORY VERIFIED")
print("=" * 70)

print("\nPrototype shape:", saved_prototypes.shape)
print("Disease ordering:", saved_labels)
print("Maximum save/reload difference:", save_reload_error)

print("\nSUCCESS: Saved prototype memory is identical to the constructed memory.")
print("=" * 70)

SAVED PROTOTYPE MEMORY VERIFIED

Prototype shape: torch.Size([8, 768])
Disease ordering: ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
Maximum save/reload difference: 0.0

SUCCESS: Saved prototype memory is identical to the constructed memory.



###### CELL 9 — NORMALIZE DISEASE PROTOTYPES FOR COSINE ANALYSIS


The prototype vectors may have different magnitudes.

To analyse whether two prototypes point in similar or different directions
in the learned 768-dimensional feature space, we normalize every prototype
to unit length.

For prototype Pₖ:

    P̂ₖ = Pₖ / ||Pₖ||₂

After normalization:

    cosine_similarity(P̂ₐ, P̂ᵦ)

measures directional similarity independent of vector magnitude.

IMPORTANT:

This normalization is for analysis only.

We are NOT replacing the saved Disease Prototype Memory.

The original [8, 768] prototypes remain unchanged and will be used later
according to the architecture.

The normalized copies are only used to investigate prototype relationships.


---

In [ ]:
# ============================================================
# CELL 9 — NORMALIZE DISEASE PROTOTYPES FOR COSINE ANALYSIS
# ============================================================

prototype_norms = torch.norm(
    saved_prototypes,
    p=2,
    dim=1
)

normalized_prototypes = F.normalize(
    saved_prototypes,
    p=2,
    dim=1
)

normalized_norms = torch.norm(
    normalized_prototypes,
    p=2,
    dim=1
)

prototype_norm_df = pd.DataFrame({
    "Disease": LABEL_COLUMNS,
    "Original_L2_Norm": prototype_norms.numpy(),
    "Normalized_L2_Norm": normalized_norms.numpy()
})

assert torch.allclose(
    normalized_norms,
    torch.ones(NUM_CLASSES),
    atol=1e-6
)

print("=" * 70)
print("PROTOTYPES NORMALIZED FOR COSINE ANALYSIS")
print("=" * 70)

display(prototype_norm_df)

print(
    "\nSUCCESS: All normalized prototypes have unit L2 norm."
)
print("=" * 70)

PROTOTYPES NORMALIZED FOR COSINE ANALYSIS


,Disease,Original_L2_Norm,Normalized_L2_Norm
0,N,4.154717,1.0
1,D,4.204853,1.0
2,G,3.787240,1.0
3,C,3.739744,1.0
4,A,4.197772,1.0
5,H,4.220326,1.0
6,M,3.965842,1.0
7,O,4.101030,1.0



SUCCESS: All normalized prototypes have unit L2 norm.


###### CELL 10 — CALCULATE THE FULL PROTOTYPE SIMILARITY MATRIX


We now compare every disease prototype against every other prototype.

Using normalized prototypes:

    Similarity = P̂ × P̂ᵀ

This produces an 8 × 8 matrix.

Interpretation:

    1.0   → identical direction
    0.0   → unrelated / orthogonal direction
   -1.0   → opposite direction

The diagonal must be approximately 1 because every prototype is compared
with itself.

The off-diagonal values show how similar the disease representations are
in the learned feature space.

This does NOT require every off-diagonal value to be near zero.

Some diseases may legitimately share visual patterns or co-occur in
multi-label images. However, if all prototypes are almost identical,
that would indicate poor separation in the learned feature space.


---

In [ ]:
# ============================================================
# CELL 10 — CALCULATE THE FULL PROTOTYPE SIMILARITY MATRIX
# ============================================================

prototype_similarity = (
    normalized_prototypes
    @ normalized_prototypes.T
)

# Numerical symmetry check
assert torch.allclose(
    prototype_similarity,
    prototype_similarity.T,
    atol=1e-6
)

# Diagonal self-similarity check
assert torch.allclose(
    torch.diag(prototype_similarity),
    torch.ones(NUM_CLASSES),
    atol=1e-5
)

prototype_similarity_df = pd.DataFrame(
    prototype_similarity.numpy(),
    index=LABEL_COLUMNS,
    columns=LABEL_COLUMNS
)

print("=" * 70)
print("DISEASE PROTOTYPE COSINE SIMILARITY MATRIX")
print("=" * 70)

display(
    prototype_similarity_df.round(4)
)

print("\nSUCCESS: Full 8 × 8 prototype similarity matrix calculated.")
print("=" * 70)

DISEASE PROTOTYPE COSINE SIMILARITY MATRIX


,N,D,G,C,A,H,M,O
N,1.0000,0.9971,0.9740,0.8203,0.9928,0.9932,0.9367,0.9978
D,0.9971,1.0000,0.9657,0.8055,0.9929,0.9974,0.9325,0.9971
G,0.9740,0.9657,1.0000,0.8987,0.9740,0.9601,0.9642,0.9803
C,0.8203,0.8055,0.8987,1.0000,0.8211,0.7873,0.8563,0.8407
A,0.9928,0.9929,0.9740,0.8211,1.0000,0.9886,0.9567,0.9957
H,0.9932,0.9974,0.9601,0.7873,0.9886,1.0000,0.9255,0.9928
M,0.9367,0.9325,0.9642,0.8563,0.9567,0.9255,1.0000,0.9491
O,0.9978,0.9971,0.9803,0.8407,0.9957,0.9928,0.9491,1.0000



SUCCESS: Full 8 × 8 prototype similarity matrix calculated.


CELL 11 — ANALYSE PROTOTYPE UNIQUENESS AND SEPARATION


The full 8 × 8 cosine similarity matrix has been calculated.

We now summarize the off-diagonal relationships between prototypes.

Self-similarity on the diagonal is excluded because every normalized
prototype has similarity 1.0 with itself.

For each disease, we identify:

    • the most similar other disease prototype
    • the corresponding cosine similarity

We also calculate overall minimum, maximum and mean off-diagonal
similarity.

Because the current PyTorch environment does not support torch.nanmax(),
we use -infinity on the diagonal instead. This safely excludes
self-similarity while allowing torch.max() to find the most similar
OTHER prototype.

This analysis is important because the prototype similarity matrix already
suggests that several disease prototypes occupy very similar directions in
the learned feature space.

We will not modify the prototypes here. We are only diagnosing the
representations before proceeding to Disease-Aware Attention.
---

In [ ]:
# ============================================================
# CELL 11 — ANALYSE PROTOTYPE UNIQUENESS AND SEPARATION
# ============================================================

# Clone the similarity matrix so the original remains unchanged
off_diagonal_similarity = prototype_similarity.clone()

# Replace diagonal self-similarity with -infinity
# This ensures each prototype cannot select itself as its
# "most similar other disease".
off_diagonal_similarity.fill_diagonal_(float("-inf"))

# Find the most similar OTHER prototype for every disease
most_similar_values, most_similar_indices = torch.max(
    off_diagonal_similarity,
    dim=1
)

most_similar_diseases = [
    LABEL_COLUMNS[index.item()]
    for index in most_similar_indices
]

separation_df = pd.DataFrame({
    "Disease": LABEL_COLUMNS,
    "Most_Similar_Other_Disease": most_similar_diseases,
    "Cosine_Similarity": most_similar_values.cpu().numpy()
})

# Extract genuine off-diagonal values only
diagonal_mask = ~torch.eye(
    NUM_CLASSES,
    dtype=torch.bool
)

valid_off_diagonal = prototype_similarity[
    diagonal_mask
]

print("=" * 70)
print("PROTOTYPE UNIQUENESS ANALYSIS")
print("=" * 70)

display(
    separation_df.sort_values(
        "Cosine_Similarity",
        ascending=False
    )
)

print("\nOverall off-diagonal similarity statistics:")
print("Minimum:", valid_off_diagonal.min().item())
print("Maximum:", valid_off_diagonal.max().item())
print("Mean:", valid_off_diagonal.mean().item())

print("\nSUCCESS: Prototype diversity statistics calculated.")
print("=" * 70)

PROTOTYPE UNIQUENESS ANALYSIS


,Disease,Most_Similar_Other_Disease,Cosine_Similarity
0,N,O,0.997769
7,O,N,0.997769
5,H,D,0.997386
1,D,H,0.997386
4,A,O,0.995652
2,G,O,0.980311
6,M,G,0.964182
3,C,G,0.898694



Overall off-diagonal similarity statistics:
Minimum: 0.7873468399047852
Maximum: 0.9977694749832153
Mean: 0.9390654563903809

SUCCESS: Prototype diversity statistics calculated.


 CELL 12 — CALCULATE FEATURE-TO-PROTOTYPE SIMILARITIES


The prototype similarity analysis compares prototype against prototype.

Now we perform a more meaningful test:

    How similar is each individual learned feature to every disease
    prototype?

We normalize all 4,491 trained feature vectors and calculate:

    Feature Similarity Matrix
        =
    normalized_features × normalized_prototypesᵀ

This produces:

    [4491, 8]

For every training eye, we now have its cosine similarity to:

    N prototype
    D prototype
    G prototype
    C prototype
    A prototype
    H prototype
    M prototype
    O prototype

These similarities are directly relevant to the next Disease-Aware
Attention stage, where image features will interact with the prototype
memory.


---

In [ ]:
# ============================================================
# CELL 12 — CALCULATE FEATURE-TO-PROTOTYPE SIMILARITIES
# ============================================================

# Normalize trained FiLM-conditioned image features
normalized_train_features = F.normalize(
    train_features,
    p=2,
    dim=1
)

# [4491, 768] @ [768, 8] = [4491, 8]
feature_prototype_similarity = (
    normalized_train_features
    @ normalized_prototypes.T
)

assert feature_prototype_similarity.shape == (
    num_samples,
    NUM_CLASSES
)

assert torch.isfinite(
    feature_prototype_similarity
).all()

print("=" * 70)
print("FEATURE-TO-PROTOTYPE SIMILARITIES CALCULATED")
print("=" * 70)

print("\nSimilarity matrix shape:")
print(feature_prototype_similarity.shape)

print("\nMinimum similarity:")
print(feature_prototype_similarity.min().item())

print("\nMaximum similarity:")
print(feature_prototype_similarity.max().item())

print(
    "\nSUCCESS: Every training feature has been compared with all 8 prototypes."
)
print("=" * 70)

FEATURE-TO-PROTOTYPE SIMILARITIES CALCULATED

Similarity matrix shape:
torch.Size([4491, 8])

Minimum similarity:
0.41941314935684204

Maximum similarity:
0.9702831506729126

SUCCESS: Every training feature has been compared with all 8 prototypes.


CELL 13 — MEASURE OWN-DISEASE PROTOTYPE AFFINITY


We now use the ground-truth multi-label information to answer:

    Do disease-positive features tend to show meaningful similarity
    to the prototype constructed from that disease?

For every disease k, we select all training eyes where:

    yᵢₖ = 1

and collect their similarity to:

    Pₖ

We calculate:

    • mean similarity
    • median similarity
    • minimum similarity
    • maximum similarity

This is not expected to be a perfect score.

Each disease contains visual variation, and some samples are multi-label.

However, these statistics tell us whether the prototype is meaningfully
connected to the features that contributed to its construction.

---

In [ ]:
# ============================================================
# CELL 13 — MEASURE OWN-DISEASE PROTOTYPE AFFINITY
# ============================================================

own_affinity_results = []

for disease_index, disease_name in enumerate(LABEL_COLUMNS):

    disease_mask = (
        train_labels[:, disease_index] == 1
    )

    own_similarities = feature_prototype_similarity[
        disease_mask,
        disease_index
    ]

    own_affinity_results.append({
        "Disease": disease_name,
        "Samples": int(disease_mask.sum().item()),
        "Mean_Own_Similarity": own_similarities.mean().item(),
        "Median_Own_Similarity": own_similarities.median().item(),
        "Min_Own_Similarity": own_similarities.min().item(),
        "Max_Own_Similarity": own_similarities.max().item()
    })

own_affinity_df = pd.DataFrame(
    own_affinity_results
)

print("=" * 70)
print("OWN-DISEASE PROTOTYPE AFFINITY")
print("=" * 70)

display(
    own_affinity_df.round(4)
)

print(
    "\nSUCCESS: Ground-truth disease features were evaluated against "
    "their corresponding prototypes."
)
print("=" * 70)

OWN-DISEASE PROTOTYPE AFFINITY


,Disease,Samples,Mean_Own_Similarity,Median_Own_Similarity,Min_Own_Similarity,Max_Own_Similarity
0,N,1502,0.8870,0.9040,0.5594,0.9633
1,D,1491,0.8903,0.9105,0.4651,0.9703
2,G,269,0.8578,0.8708,0.6503,0.9413
3,C,286,0.8475,0.8644,0.6202,0.9431
4,A,215,0.8859,0.8998,0.6076,0.9684
5,H,140,0.8976,0.9134,0.5431,0.9581
6,M,226,0.8598,0.8746,0.5418,0.9469
7,O,1089,0.8775,0.9012,0.4724,0.9601



SUCCESS: Ground-truth disease features were evaluated against their corresponding prototypes.


CELL 14 — COMPARE OWN-DISEASE AFFINITY WITH NON-LABEL PROTOTYPES


High similarity to an own prototype alone is not enough because feature
representations may generally be similar to many prototypes.

Therefore, for each disease-positive feature, we compare:

    similarity to its own disease prototype

against:

    similarity to prototypes whose labels are NOT positive for that eye

For multi-label eyes, all of their true disease prototypes are excluded
from the "non-label" comparison.

This gives a cleaner test of prototype relevance.

For each disease, we calculate:

    Mean Own-Label Similarity
    Mean Non-Label Similarity
    Separation = Own - Non-Label

A positive separation indicates that, on average, features belonging to
that disease are more aligned with their true disease prototype than with
prototypes of diseases they do not have.

This is one of our strongest checks before moving toward Disease-Aware
Attention.
---

In [ ]:
# ============================================================
# CELL 14 — COMPARE OWN-DISEASE AFFINITY WITH NON-LABEL PROTOTYPES
# ============================================================

separation_results = []

for disease_index, disease_name in enumerate(LABEL_COLUMNS):

    disease_mask = (
        train_labels[:, disease_index] == 1
    )

    selected_similarities = feature_prototype_similarity[
        disease_mask
    ]

    selected_labels = train_labels[
        disease_mask
    ]

    # Similarity to this disease's prototype
    own_similarity = selected_similarities[
        :,
        disease_index
    ]

    # Prototypes corresponding to any true label are excluded
    non_label_mask = selected_labels == 0

    non_label_values = selected_similarities[
        non_label_mask
    ]

    mean_own = own_similarity.mean().item()
    mean_non_label = non_label_values.mean().item()
    separation = mean_own - mean_non_label

    separation_results.append({
        "Disease": disease_name,
        "Mean_Own_Label_Similarity": mean_own,
        "Mean_Non_Label_Similarity": mean_non_label,
        "Separation": separation
    })

prototype_relevance_df = pd.DataFrame(
    separation_results
)

print("=" * 70)
print("PROTOTYPE RELEVANCE SEPARATION")
print("=" * 70)

display(
    prototype_relevance_df.round(4)
)

print(
    "\nPositive separation means true disease features are, on average, "
    "more similar to their own prototype than to non-label prototypes."
)

print("=" * 70)

PROTOTYPE RELEVANCE SEPARATION


,Disease,Mean_Own_Label_Similarity,Mean_Non_Label_Similarity,Separation
0,N,0.8870,0.8518,0.0352
1,D,0.8903,0.8506,0.0396
2,G,0.8578,0.8210,0.0368
3,C,0.8475,0.7028,0.1447
4,A,0.8859,0.8510,0.0349
5,H,0.8976,0.8496,0.0480
6,M,0.8598,0.8005,0.0593
7,O,0.8775,0.8459,0.0316



Positive separation means true disease features are, on average, more similar to their own prototype than to non-label prototypes.


 CELL 15 — SAVE PROTOTYPE ANALYSIS RESULTS


The Disease Prototype Memory has been constructed and analysed.

Before closing this notebook, we save the important diagnostic results
alongside the prototype memory.

These results document:

    • Prototype similarity relationships
    • Prototype uniqueness statistics
    • Own-disease prototype affinity
    • Prototype relevance separation

Saving these results allows us to preserve the diagnostic evidence without
recomputing the analysis in later notebooks.

The prototype vectors themselves remain stored separately in:

    disease_prototypes.pt
---

In [17]:
# ============================================================
# CELL 15 — SAVE PROTOTYPE ANALYSIS RESULTS
# ============================================================

ANALYSIS_SAVE_PATH = os.path.join(
    ROOT,
    "prototype_analysis.pt"
)

prototype_analysis = {
    "label_columns": LABEL_COLUMNS,

    "prototype_similarity": prototype_similarity.cpu(),

    "prototype_counts": prototype_counts.cpu(),

    "own_affinity": own_affinity_df.to_dict(
        orient="records"
    ),

    "prototype_relevance": prototype_relevance_df.to_dict(
        orient="records"
    ),

    "uniqueness_analysis": separation_df.to_dict(
        orient="records"
    ),

    "off_diagonal_min": valid_off_diagonal.min().item(),
    "off_diagonal_max": valid_off_diagonal.max().item(),
    "off_diagonal_mean": valid_off_diagonal.mean().item()
}

torch.save(
    prototype_analysis,
    ANALYSIS_SAVE_PATH
)

assert os.path.exists(ANALYSIS_SAVE_PATH)

print("Prototype analysis saved successfully.")
print("Path:", ANALYSIS_SAVE_PATH)

Prototype analysis saved successfully.
Path: /content/drive/My Drive/Eye Disease/Dataset/prototype_analysis.pt


 CELL 16 — FINAL PROTOTYPE MEMORY INTEGRITY CHECK


Before moving to Disease-Aware Attention, we perform one final integrity
check on the saved Disease Prototype Memory.

We confirm:

    1. Exactly 8 disease prototypes exist.
    2. Every prototype has 768 dimensions.
    3. All values are finite.
    4. Disease ordering is correct.
    5. Every disease has contributed training samples.
    6. The saved analysis corresponds to the same disease ordering.
    7. All prototype relevance separations are positive.

This does not require the prototypes to be perfectly separated.

The purpose is to confirm that the prototype memory is structurally valid
and disease-relevant before it becomes the memory input to the next
Disease-Aware Attention module.


In [18]:
# ============================================================
# CELL 16 — FINAL PROTOTYPE MEMORY INTEGRITY CHECK
# ============================================================

reloaded_analysis = torch.load(
    ANALYSIS_SAVE_PATH,
    map_location="cpu",
    weights_only=False
)

final_prototypes = saved_prototypes

all_separations = torch.tensor(
    prototype_relevance_df["Separation"].values,
    dtype=torch.float32
)

integrity_checks = {
    "Correct prototype shape":
        tuple(final_prototypes.shape)
        == (NUM_CLASSES, FEATURE_DIM),

    "All prototype values finite":
        torch.isfinite(final_prototypes).all().item(),

    "Correct disease ordering":
        saved_labels == LABEL_COLUMNS,

    "All diseases have samples":
        torch.all(prototype_counts > 0).item(),

    "Analysis disease ordering matches":
        reloaded_analysis["label_columns"]
        == LABEL_COLUMNS,

    "All disease separations positive":
        torch.all(all_separations > 0).item()
}

integrity_df = pd.DataFrame(
    list(integrity_checks.items()),
    columns=["Check", "Passed"]
)

display(integrity_df)

assert all(integrity_checks.values())

print("\n" + "=" * 70)
print("FINAL INTEGRITY CHECK PASSED")
print("=" * 70)

print(
    "\nDisease Prototype Memory is ready for "
    "Disease-Aware Attention."
)

,Check,Passed
0,Correct prototype shape,True
1,All prototype values finite,True
2,Correct disease ordering,True
3,All diseases have samples,True
4,Analysis disease ordering matches,True
5,All disease separations positive,True



FINAL INTEGRITY CHECK PASSED

Disease Prototype Memory is ready for Disease-Aware Attention.


 CELL 17 — DISEASE PROTOTYPE MEMORY FINAL SUMMARY


Notebook 2 is now complete.

Starting from the trained FiLM-conditioned feature representations:

    train_features.pt
        ↓
    [4491, 768] learned image features
        +
    [4491, 8] multi-label ground truth
        ↓
    Disease-wise feature averaging
        ↓
    [8, 768] Disease Prototype Memory

The prototype memory has been:

    • Constructed
    • Independently verified
    • Saved and reloaded
    • Analysed using cosine similarity
    • Tested against individual disease features
    • Evaluated for own-disease affinity
    • Evaluated against non-label prototypes
    • Passed the final integrity check

The resulting files are now ready for Notebook 3:

    disease_prototypes.pt
    prototype_analysis.pt

Notebook 3 will load the prototype memory and integrate it into the
Disease-Aware Attention module.
---